In [1]:
!python --version

Python 3.12.0


In [2]:
import ultralytics
ultralytics.__version__

'8.3.102'

In [3]:
import torch
torch.__version__

'2.6.0+cpu'

In [4]:
import torch

if torch.cuda.is_available():
    print("CUDA is available! GPU:", torch.cuda.get_device_name(0))
    device = 'cuda'
else:
    print("CUDA not available — running on CPU")
    device = 'cpu'


CUDA not available — running on CPU


# Detect, track and count Persons

In [5]:
%cd yolov8_DeepSORT

[WinError 2] The system cannot find the file specified: 'yolov8_DeepSORT'
c:\Users\kg956\Desktop\Tracking-and-counting-Using-YOLOv8-and-DeepSORT-main\Tracking-and-counting-Using-YOLOv8-and-DeepSORT-main


In [1]:
from ultralytics import YOLO

import time
import torch
import cv2
import torch.backends.cudnn as cudnn
from PIL import Image
import colorsys
import numpy as np

# Load a model
model = YOLO("yolov8n.pt")  # load a pretrained model (recommended for training)

results = model("images/img.jpg", save=True,show_conf=False)



class_names = ['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

for result in results:
    boxes = result.boxes  # Boxes object for bbox outputs
    probs = result.probs  # Class probabilities for classification outputs
    cls = boxes.cls.tolist()  # Convert tensor to list
    xyxy = boxes.xyxy
    xywh = boxes.xywh  # box with xywh format, (N, 4)
    conf = boxes.conf
    print(cls)
    for class_index in cls:
        class_name = class_names[int(class_index)]
        print("Class:", class_name)


image 1/1 c:\Users\kg956\Desktop\Tracking-and-counting-Using-YOLOv8-and-DeepSORT-main\Tracking-and-counting-Using-YOLOv8-and-DeepSORT-main\images\img.jpg: 384x640 2 persons, 1 car, 1 dog, 439.3ms
Speed: 16.3ms preprocess, 439.3ms inference, 33.5ms postprocess per image at shape (1, 3, 384, 640)
Results saved to runs\detect\predict2
[16.0, 0.0, 0.0, 2.0]
Class: dog
Class: person
Class: person
Class: car


# DeepSORT

In [7]:
from deep_sort.utils.parser import get_config
from deep_sort.deep_sort import DeepSort
from deep_sort.sort.tracker import Tracker

deep_sort_weights = 'deep_sort/deep/checkpoint/ckpt.t7'
tracker = DeepSort(model_path=deep_sort_weights, max_age=70)

In [7]:
# Define the video path
video_path = 'test_videos/2.mp4'

cap = cv2.VideoCapture(video_path)

# Get the video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))


# Define the codec and create VideoWriter object
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_path = 'output6.mp4'
out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
frames = []

unique_track_ids = set()

In [1]:
import cv2
import time
import torch
from ultralytics import YOLO

# ================================
# Load YOLOv8 model
# ================================
model = YOLO("yolov8n.pt")
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'

# ================================
# COCO class names
# ================================
coco_classes = [
    'person','car','Car','bus','train','Truck','Scooty','boat',
    'traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat','dog',
    'horse','sheep','cow','elephant','bear','zebra','giraffe','Lion','backpack',
    'handbag','tie','suitcase','frisbee','skis','snowboard','sports ball',
    'baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle',
    'wine glass','cup','fork','Knife','spoon','bowl','banana','apple','sandwich','orange',
    'broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant',
    'bed','dining table','toilet','tv','Laptop','mouse','remote','Key-Board','cell phone',
    'microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors',
    'teddy bear','hair drier','toothbrush'
]

# ================================
# Video input/output
# ================================
video_path = 'test_videos/7.mp4'
cap = cv2.VideoCapture(video_path)

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps_input = cap.get(cv2.CAP_PROP_FPS)

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
output_path = 'output_all_classes.mp4'
out = cv2.VideoWriter(output_path, fourcc, fps_input, (frame_width, frame_height))

# ================================
# Tracking variables
# ================================
unique_track_ids = set()
counter, fps, elapsed = 0, 0, 0
start_time = time.perf_counter()

# Accuracy variables
total_frames = 0
correct_detections = 0

print("⏳ Processing video with YOLOv8 + ByteTrack for all classes...")

# ================================
# Main loop
# ================================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    total_frames += 1

    # Run YOLOv8 tracking (all classes)
    results = model.track(frame, tracker="bytetrack.yaml", conf=0.5)
    
    detected = False  # Flag to count correct frame
    for box in results[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        track_id = int(box.id)
        class_id = int(box.cls[0].cpu().numpy())
        class_name = coco_classes[class_id]

        # Draw bounding box and label
        color = ((track_id*37)%255, (track_id*17)%255, (track_id*29)%255)
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        cv2.putText(frame, f"{class_name}-{track_id}", (int(x1), int(y1)-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.9, color, 4)

        unique_track_ids.add(track_id)
        detected = True  # Object detected in this frame

    if detected:
        correct_detections += 1  # Count this frame as correct

    # ================================
    # Counting & FPS
    # ================================
    object_count = len(unique_track_ids)
    counter += 1
    current_time = time.perf_counter()
    elapsed = current_time - start_time
    if elapsed > 1:
        fps = counter / elapsed
        counter = 0
        start_time = current_time

    # Calculate accuracy in real-time
    accuracy = (correct_detections / total_frames) * 100

    # Overlay info
    cv2.putText(frame, f"Tracked Objects: {object_count}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    cv2.putText(frame, f"FPS: {fps:.2f}", (10, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
    cv2.putText(frame, f"Accuracy: {accuracy:.2f}%", (10, 110),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # Write frame to output
    out.write(frame)

# ================================
# Cleanup
# ================================
cap.release()
out.release()
cv2.destroyAllWindows()

print("✅ Done! Output saved to:", output_path)
print(f"Total unique objects tracked: {len(unique_track_ids)}")
print(f"Final Accuracy: {accuracy:.2f}%")

⏳ Processing video with YOLOv8 + ByteTrack for all classes...

0: 384x640 8 persons, 5 cars, 130.7ms
Speed: 13.4ms preprocess, 130.7ms inference, 8.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 5 cars, 74.6ms
Speed: 2.7ms preprocess, 74.6ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 6 cars, 73.2ms
Speed: 2.3ms preprocess, 73.2ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 6 cars, 87.0ms
Speed: 2.1ms preprocess, 87.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 5 cars, 69.5ms
Speed: 2.0ms preprocess, 69.5ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 6 cars, 65.8ms
Speed: 2.2ms preprocess, 65.8ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 7 persons, 4 cars, 84.9ms
Speed: 1.9ms preprocess, 84.9ms inference, 1.0ms postprocess per image at shape (1, 3, 